# Keyframe Extraction with TransNetV2 Shot Detection

Detects shot boundaries via TransNetV2 (CPU), extracts 3 keyframes per shot at 15%/50%/85% positions,
saves lossless PNGs named by frame index for downstream OCR.

**Prerequisites:** Python 3.12, ffmpeg in PATH, `nix profile install nixpkgs#ffmpeg` if needed.

**Before running:** `Kernel → Restart & Clear Outputs`

In [ ]:
# Dependencies are managed by uv (pyproject.toml + `uv sync` in flake.nix shellHook).
# No pip install needed here.


In [ ]:
# Verify TransNetV2 weights are real (not LFS pointers). Reinstall if corrupted.
import os
weights_dir = os.path.join(os.path.dirname(__import__("transnetv2").__file__),
                          "transnetv2-weights")
weights_pb = os.path.join(weights_dir, "saved_model.pb")
if os.path.getsize(weights_pb) < 1000:
    raise RuntimeError(
        "TransNetV2 weights are LFS pointer stubs. Run:\n"
        "  git clone --depth 1 https://github.com/soCzech/TransNetV2.git /tmp/tv2 &&\n"
        "  cd /tmp/tv2 && git lfs install --local && git lfs pull &&\n"
        f"  cp inference/transnetv2-weights/* {weights_dir} &&\n"
        f"  cp inference/transnetv2-weights/variables/* {weights_dir}/variables/")
print("TransNetV2 weights OK")


In [ ]:
import os, subprocess, shutil, time, json
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed, ThreadPoolExecutor
import numpy as np
import cv2
from tqdm.auto import tqdm

if not Path("pyproject.toml").exists() and (Path.cwd().parent / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# --- Config ---
VIDEOS_DIR = Path("data/2026/videos")
OUTPUT_DIR = Path("data/2026/keyframes")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

POSITIONS = [0.15, 0.5, 0.85]
SHOT_THRESHOLD = 0.5
MIN_SHOT_LEN = 15
MAX_WORKERS = 3
SKIP_EXISTING = True

if shutil.which("ffmpeg") is None:
    raise RuntimeError("ffmpeg not found. Install: nix profile install nixpkgs#ffmpeg")

videos = sorted(VIDEOS_DIR.rglob("*.mp4"))
print(f"Found {len(videos)} videos in {OUTPUT_DIR.resolve()}")

In [ ]:
def get_video_fps(video_path):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    return fps if fps > 0 else 30.0


def detect_shots(model, video_path, threshold=SHOT_THRESHOLD, min_shot_len=MIN_SHOT_LEN):
    _, preds, _ = model.predict_video(str(video_path))
    preds = preds.squeeze()
    n_frames = len(preds)

    above = preds > threshold
    cuts = [0]
    i = 0
    while i < n_frames:
        if above[i]:
            j = i + 1
            while j < n_frames and above[j]:
                j += 1
            peak = i + int(np.argmax(preds[i:j]))
            if peak - cuts[-1] >= min_shot_len:
                cuts.append(peak)
            i = j
        else:
            i += 1

    shots = [(cuts[k], cuts[k + 1] - 1) for k in range(len(cuts) - 1)]
    if cuts[-1] < n_frames - 1:
        shots.append((cuts[-1], n_frames - 1))
    return [(s, e) for s, e in shots if e - s >= min_shot_len]


def compute_keyframes(shots, positions=POSITIONS):
    indices = []
    for start, end in shots:
        for pos in positions:
            idx = int(start + (end - start) * pos)
            indices.append(max(start, min(end, idx)))
    return indices


def _extract_one(args):
    video_path, timestamp, out_path = args
    subprocess.run([
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-ss", str(timestamp), "-i", str(video_path),
        "-frames:v", "1", "-c:v", "png",
        "-pix_fmt", "rgb24", str(out_path),
    ], check=True, capture_output=True, timeout=30)


def extract_frames(video_path, frame_indices, output_dir, fps):
    output_dir.mkdir(parents=True, exist_ok=True)
    tasks = []
    for fidx in frame_indices:
        out_path = output_dir / f"{fidx:06d}.png"
        if not out_path.exists():
            tasks.append((str(video_path), fidx / fps, out_path))
    if tasks:
        with ThreadPoolExecutor(max_workers=min(8, len(tasks))) as pool:
            list(pool.map(_extract_one, tasks))

In [ ]:
def process_video(video_path):
    global _model
    rel = video_path.relative_to(VIDEOS_DIR)
    out_dir = OUTPUT_DIR / rel.with_suffix("")
    done_file = out_dir / "_done"

    if SKIP_EXISTING and done_file.exists():
        return {"video": str(rel), "skipped": True}

    out_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    shots = detect_shots(_model, str(video_path))
    t1 = time.perf_counter()

    if not shots:
        done_file.write_text("")
        return {"video": str(rel), "shots": 0, "frames": 0, "detect_s": t1 - t0}

    frame_indices = compute_keyframes(shots)
    fps = get_video_fps(str(video_path))
    extract_frames(str(video_path), frame_indices, out_dir, fps)
    t2 = time.perf_counter()

    done_file.write_text(json.dumps({"shots": len(shots), "frames": len(frame_indices)}))
    return {"video": str(rel), "shots": len(shots), "frames": len(frame_indices),
            "detect_s": t1 - t0, "extract_s": t2 - t1, "duration": t2 - t0}

In [ ]:
def init_worker():
    """Load TransNetV2 once per worker process."""
    global _model
    import tensorflow as tf
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    from transnetv2 import TransNetV2
    _model = TransNetV2()

In [ ]:
# --- Parallel extraction ---
t_start = time.perf_counter()
results = []

with ProcessPoolExecutor(max_workers=MAX_WORKERS, initializer=init_worker) as executor:
    futures = {executor.submit(process_video, v): v for v in videos}
    pbar = tqdm(as_completed(futures), total=len(videos), desc="Videos", unit="vid")
    for future in pbar:
        try:
            r = future.result()
            results.append(r)
            if not r.get("skipped"):
                pbar.set_postfix_str(f"{r.get('shots',0)} shots")
        except Exception as e:
            vname = futures[future].relative_to(VIDEOS_DIR)
            results.append({"video": str(vname), "error": str(e)})
            pbar.write(f"  FAIL {vname}: {e}")

t_total = time.perf_counter() - t_start

In [ ]:
# --- Summary ---
total_shots = sum(r.get("shots", 0) for r in results if "error" not in r)
total_frames = sum(r.get("frames", 0) for r in results if "error" not in r)
skipped = sum(1 for r in results if r.get("skipped"))
errors = [r for r in results if "error" in r]
processed = len(results) - skipped - len(errors)

print(f"{'='*60}")
print(f"Total:  {t_total/60:.1f} min ({t_total:.0f}s)")
print(f"Videos: {len(videos)} ({processed} done, {skipped} cached, {len(errors)} errors)")
print(f"Shots:  {total_shots} ({total_shots/max(1, processed):.0f}/vid avg)")
print(f"Frames: {total_frames} ({total_frames/max(1, processed):.0f}/vid avg)")
print(f"{'='*60}")
if errors:
    print(f"\nFailed ({len(errors)}):")
    for r in errors:
        print(f"  {r['video']}: {r['error']}")

In [ ]:
# --- Recover empty (0-byte) keyframes by re-extracting from source videos ---
import subprocess, os, cv2
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

VIDEOS = Path("data/2026/videos")
KEYFRAMES = Path("data/2026/keyframes")

def find_empty():
    for dirpath, dirs, files in os.walk(KEYFRAMES):
        for fn in files:
            if fn.endswith(".png"):
                p = Path(dirpath) / fn
                try:
                    if p.stat().st_size == 0:
                        yield p
                except OSError:
                    pass

empty = list(find_empty())
print(f"{len(empty)} empty keyframes")

if empty:
    fps_cache = {}
    for rel in {p.parent.relative_to(KEYFRAMES) for p in empty}:
        cap = cv2.VideoCapture(str(VIDEOS / f"{rel}.mp4"))
        fps = cap.get(cv2.CAP_PROP_FPS)
        cap.release()
        fps_cache[rel] = fps if fps > 0 else 30.0

    def recover(p):
        rel = p.parent.relative_to(KEYFRAMES)
        fidx = int(p.stem)
        tmp = p.with_suffix(".tmp.png")
        subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
                        "-ss", str(fidx / fps_cache[rel]), "-i", str(VIDEOS / f"{rel}.mp4"),
                        "-frames:v", "1", "-c:v", "png", str(tmp)],
                       check=True, capture_output=True, timeout=60)
        tmp.replace(p)

    with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
        list(tqdm(pool.map(recover, empty), total=len(empty), desc="Recover", unit="img"))


In [ ]:
# --- Re-encode existing keyframes with default PNG compression ---
# Lossless: only shrinks file size. Original files were saved with
# -compression_level 0 (zlib "stored", header 0x78 0x01); already re-encoded
# files use 0x78 0x9c, so we skip them and can safely resume after a crash.
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import subprocess, os
from tqdm.auto import tqdm

ROOT = Path("data/2026/keyframes")

def is_uncompressed(p):
    with open(p, "rb") as fh:
        d = fh.read(4096)
    i = 8
    while i + 8 <= len(d):
        ln = int.from_bytes(d[i:i+4], "big")
        if d[i+4:i+8] == b"IDAT":
            return d[i+8:i+10] == b"\x78\x01"
        i += 12 + ln
    return True  # unknown -> re-encode (idempotent)

def reencode(p):
    tmp = p.with_suffix(".tmp.png")
    try:
        subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
                        "-i", str(p), "-c:v", "png", str(tmp)],
                       check=True, capture_output=True)
        tmp.replace(p)
    except Exception as e:
        tqdm.write(f"  FAIL {p}: {e}")

files = [p for p in ROOT.rglob("*.png") if not p.name.endswith(".tmp.png")]
todo = [p for p in tqdm(files, desc="Scan", unit="img") if is_uncompressed(p)]
print(f"{len(todo)} / {len(files)} cần re-encode")

with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
    list(tqdm(pool.map(reencode, todo), total=len(todo), desc="Re-encode", unit="img"))


In [ ]:
# --- Verify all keyframes are valid PNGs ---
from pathlib import Path
from tqdm.auto import tqdm

ROOT = Path("data/2026/keyframes")
SIG = b"\x89PNG\r\n\x1a\n"

broken = []
total = 0
for p in tqdm(ROOT.rglob("*.png"), desc="Verify", unit="img"):
    if p.name.endswith(".tmp.png"):
        continue
    total += 1
    try:
        with open(p, "rb") as fh:
            head = fh.read(8)
            fh.seek(-12, 2)
            tail = fh.read(12)
        if head != SIG or len(tail) < 12 or tail[4:8] != b"IEND":
            broken.append(p)
    except OSError:
        broken.append(p)

print(f"Checked {total} keyframes")
print(f"Broken: {len(broken)}")
for p in broken:
    print("  ", p)
